# Python-1, Лекция 8

Лектор: Хайбулин Даниэль

Подготовил материал: Хайбулин Даниэль

Сегодня мы поговорим о протоколах и интерфейсах в питоне.

## Утиная типизация

<div style="
    background-color: #44944A;
    padding: 15px;
    border-radius: 10px;
    border: 1px solid #fbfbfbff;
    text-align: center;
    display: flex;
    justify-content: center;
    align-items: center;
    min-height: 15px;
">
    <span style="color: white; font-weight: bold;">
        Если оно разговаривает как утка и ходит как утка, то, скорее всего, это утка!
    </span>
</div>

<div style="
    background-color: #44944A;
    padding: 15px;
    border-radius: 10px;
    border: 1px solid #fbfbfbff;
    text-align: center;
    display: flex;
    justify-content: center;
    align-items: center;
    min-height: 15px;
">
    <span style="color: white; font-weight: bold;">
        <strong>Протокол</strong> - неформальный интерфейс, определенный наличием конкретных методов и атрибутов.
    </span>
</div>

Чтобы реализовать пользовательский класс, который будет работать как последовательность, не нужно наследоваться от чего-либо: достаточно реализовать методы, которые прописаны в протоколе последовательности.

Протокол определен лишь в документации, но не в коде.

Пример для понимания:

Протокол последовательность - наличие dunder методов `__len__` и `__getitem__`. Всякий класс, имеющий эти методы со стандартной сигнатурой, может быть использован в том коде, в котором ожидается последовательность.

In [ ]:
import collections

Card = collections.namedtuple("Card", ["rank", "suit"])


class Deck:
    ranks = [str(n) for n in range(2, 11)] + list("JQKA")
    suits = "spades diamonds clubs hearts".split()

    def __init__(self) -> None:
        self._cards = [Card(rank, suit) for suit in self.suits for rank in self.ranks]

    def __len__(self) -> int:
        return len(self._cards)

    def __getitem__(self, position: int) -> Card:
        return self._cards[position]

Посмотрев на этот класс мы можем заявить, что это последовательность, так как он ведет себя как последовательность. Это и есть утиная типизация.

Теперь мы можем, например, пройтись циклом по нашему классу:

In [ ]:
deck = Deck()
for card in deck:
    print(card)

Заодно наш класс реализует протокол `Sized`, то есть имеет размер: возвращаемое значением метода `__len__`.

In [ ]:
len(deck)

## Monkey patching

<div style="
    background-color: #44944A;
    padding: 15px;
    border-radius: 10px;
    border: 1px solid #fbfbfbff;
    text-align: center;
    display: flex;
    justify-content: center;
    align-items: center;
    min-height: 15px;
">
    <span style="color: white; font-weight: bold;">
        Monkey patching - динамичное изменение модуля, класса или функции в рантайме.
    </span>
</div>

Давайте попробуем разобраться на примере:

In [ ]:
from random import shuffle

numbers = [_ for _ in range(10)]
print(numbers)
shuffle(numbers)
print(numbers)

Теперь хотим перемешать колоду карт:

In [ ]:
deck = Deck()
shuffle(deck)

Видим ошибку, что у нашего класса нет dunder метода `__setitem__`. Проблема в том, что `shuffle` изменяет входную коллекцию in-place, а наш `Deck` реализует протокол неизменяемой последовательности. Давайте расширим `Deck` до изменяемой последовательности:

In [ ]:
def set_card(
    self: Deck, key: int, value: Card
) -> None:  # обратите внимание как я назвал аргументы по сигнатуре
    self._cards[key] = value

In [ ]:
Deck.__setitem__ = (
    set_card  # делаем monkey patching класса, чтобы он реализовывал нужный нам протокол
)
deck = Deck()
shuffle(deck)  # теперь можно перемешать колоду карт!
for card in deck:
    print(card)

Итак, нам удалось перемешать колоду карт, определив новый метод в нашего класса `Deck`.

Выше показан динамический пример протоколов в утиной типизации. В данном случае нам все равно, есть ли у объекта изначально какие-либо методы, важно лишь одно: к моменту использования где-либо он должен реализовывать конкретный протокол.

## Гусиная типизация

Очередное животное - очередная типизация. Давайте разбираться, чем гусь отличается от утки.

Тут нам пора обуздать абстракцию, а именно: Abstract Base Class aka `ABC`. Грубо говоря это абстрактные классы, наследуясь от которых мы заявляем, что наш пользовательский класс реализует какой-либо **интерфейс** и распознается функциями `isinstance`, `issubclass`.

Утиная типизация задумана для избежания использования `isinstance` и вообще какой-либо попытки проверить с каким классом мы взаимодействуем.

Но теперь мы поиграемся с `isinstance` для понимания гусиной типизации.

In [ ]:
class MySized:
    def __len__(self) -> int:
        return 0

Теперь нам полезна стандартная библиотека `collections`, в которой есть модуль `abc`.

In [ ]:
from collections import abc

abc.__all__

И теперь через `isinstance` мы проверяем, реализовывает ли инстанс класса некоторый интерфейс, например `Sized`:

In [ ]:
assert isinstance(MySized(), abc.Sized)
assert not isinstance(MySized(), abc.Sequence)

Видим, что инстанс класса `MySized` реализовывает интерфейс `Sized`, но не реализовывает интерфейс последовательности.

Оперируя таким функционалом мы можем строже типизировать наши классы и функции, проверяя соответствие интерфейсу при помощи `isinstance` перед использованием.


<div style="
    background-color: #FFBA00;
    padding: 15px;
    border-left: 5px solid #ffcc00;
    text-align: center;
    display: flex;
    justify-content: center;
    align-items: center;
    min-height: 15px;
">
    <span style="color: white; font-weight: bold;">
        Часто встречающийся isinstance в коде - пример плохо организованного кода.
    </span>
</div>

Кроме того, люди полюбили питон за его динамичность и практичность, поэтому не стоит плодить абстракции там, где годится готовая: прежде чем писать свой абстрактный класс, проверьте, нет ли подходящего в `collections.abc`. Свои ABC тоже нужны — как их писать, разберём чуть ниже.

## Наследование от ABC

Вернемся к примеру с колодой. Реализуем ее по паттерну гусиной типизации:

In [ ]:
from collections import abc


class Deck(abc.MutableSequence[Card]):  # Наследуемся от интерфейса изменяемой последовательности.
    ...

In [ ]:
try:
    deck = Deck()
except TypeError as err:
    print(f"Не удается создать инстанс класса: {err}")

Что мы проделали тут? Мы заявили, что наш пользовательский класс `Deck` реализовывает интерфейс изменяемой последовательности, но мы не реализовали dunder методы, которые определяют изменяемую последовательность. Именно поэтому мы и не можем создать инстанс класса.

<div style="
    background-color: #8B0000;
    padding: 15px;
    border: 2px dashed #ba0606;
    border-radius: 5px;
    text-align: center;
    display: flex;
    justify-content: center;
    align-items: center;
    min-height: 20px;
">
<span style="color: white; font-weight: bold;">
        Проверка типов произошла только во время рантайма - когда пытались создать инстанс класса.
    </span>
</div>

Ну что ж, давайте тогда реализуем их:

In [ ]:
from collections import abc, namedtuple

Card = namedtuple("Card", ["rank", "suit"])


class Deck(abc.MutableSequence[Card]):
    ranks = [str(n) for n in range(2, 11)] + list("JQKA")
    suits = "spades diamonds clubs hearts".split()

    def __init__(self) -> None:
        self._cards = [Card(rank, suit) for suit in self.suits for rank in self.ranks]

    def __len__(self) -> int:
        return len(self._cards)

    def __getitem__(self, position: int) -> Card:
        return self._cards[position]

    def __setitem__(self, position: int, value: Card) -> None:
        self._cards[position] = value

    def __delitem__(self, position: int) -> None:
        del self._cards[position]

    def insert(self, position: int, value: Card) -> None:
        # тут у нас не dunder метод, но тоже заявлен в интерфейсе MutableSequence
        self._cards.insert(position, value)

In [ ]:
deck = Deck()
shuffle(deck)
for card in deck:
    print(card.rank, card.suit)

Теперь мы честно унаследовали наш класс от абстрактного интерфейса и реализовали этот интерфейс.

Ниже показана иерархия наследования абстрактных классов:

![Иерархия абстрактных классов](abc-hierarchy.png)

В нашем примере нам не нужны методы `insert` и `__delitem__` - мы проделали лишнюю работу, реализовав их. Однако к этой работе нас обязывает наследование от абстрактного класса. 

Все же, это наследование и награждает нас, от `Sequence` мы получаем следующие методы в подарок:

- `__contains__`
- `__iter__`
- `__reversed__`
- index
- count

In [ ]:
deck = Deck()

assert deck.index(Card(rank="J", suit="spades")) == 9  # можем получить индекс конкретной карты
assert (
    deck.count(Card(rank="J", suit="spades")) == 1
)  # можем посчитать оставшееся количество карт в колоде
assert deck.count(Card(rank="F", suit="spades")) == 0
assert Card(rank="J", suit="spades") in deck  # можем проверить содержится ли карта в колоде

for card in reversed(deck):  # можем развернуть колоду
    print(card.rank, card.suit)

От `MutableSequence` мы получаем методы `append`, `reverse`, `extend`, `pop`, `remove`, `__iadd__`

Возникает разумный вопрос, а как это вообще работает? Откуда наследуемые методы знают куда, например, сделать `append`?

На самом деле **append** делает что-то в духе:

```python
def append(self, value):
    self.insert(len(self), value)
```

но `insert` мы реализовали сами - это и есть магическое переиспользование того, что мы сделали, в самой функциональности языка.

Кроме последовательностей, в `collections.abc` есть ещё два больших семейства. Контракт у каждого устроен одинаково: несколько методов вы реализуете сами, остальные достаются в наследство.

| ABC | Реализовать самому | Достаётся бесплатно |
|---|---|---|
| `Sequence` | `__getitem__`, `__len__` | `__contains__`, `__iter__`, `__reversed__`, `index`, `count` |
| `MutableSequence` | плюс `__setitem__`, `__delitem__`, `insert` | плюс `append`, `extend`, `pop`, `remove`, `reverse` |
| `Mapping` | `__getitem__`, `__iter__`, `__len__` | `__contains__`, `keys`, `items`, `values`, `get`, `__eq__` |
| `MutableMapping` | плюс `__setitem__`, `__delitem__` | плюс `pop`, `popitem`, `clear`, `update`, `setdefault` |
| `Set` | `__contains__`, `__iter__`, `__len__` | `__and__`, `__or__`, `__sub__`, `__xor__`, `isdisjoint`, сравнения |
| `MutableSet` | плюс `add`, `discard` | плюс `clear`, `pop`, `remove`, `__ior__`, `__iand__` |

Логика везде та же, что мы только что разобрали: вы даёте минимальный набор операций, а базовый класс выражает через них всё остальное — как `append` выразился через ваш `insert`.

<div style="
    background-color: #8B0000;
    padding: 15px;
    border: 2px dashed #ba0606;
    border-radius: 5px;
    text-align: center;
    display: flex;
    justify-content: center;
    align-items: center;
    min-height: 20px;
">
<span style="color: white; font-weight: bold;">
        isinstance не гарантирует проверку хешируемости объекта
    </span>
</div>

С хешируемостью дела обстоят не так прозрачно как хотелось бы: проверяется лишь реализация dunder методов у класса, но не проверяется их реализация у вложенных классов. Простой знакомый нам пример:

In [ ]:
t = (1, 2, [3, 4])
assert isinstance(t, abc.Hashable)

Проверка на `isinstance` пройдена и невнимательный программист получил бы ошибки в дальнейшем, понадеявшись, что входной объект хешируемый, но мы знаем, что он нехешируемый.

<div style="
    background-color: #44944A;
    padding: 15px;
    border-radius: 10px;
    border: 1px solid #fbfbfbff;
    text-align: center;
    display: flex;
    justify-content: center;
    align-items: center;
    min-height: 15px;
">
    <span style="color: white; font-weight: bold;">
        fast fail - чем быстрее код упадет тем лучше.
    </span>
</div>

**Fast Fail** - это история о том, что лучше бы сломать код как можно быстрее. Все же самый лучший способ узнать хешируемый объект или нет - это просто вызвать от него функцию `hash`, как и гласит нам утиная типизация.

In [ ]:
try:
    hash(t)
except TypeError as err:
    print(err)

## Реализация пользовательского ABC

Вот мы разобрались с разными птичьими типизациями, наследованием от абстрактных базовых классов. Давайте теперь реализуем свой абстрактный базовый класс. В этом нам поможет наследование от `abc.ABC`. Напишем класс `Tombola` (лоттерея).

Один синтаксический момент, который встретится ниже. Запись `class Tombola[T]` означает, что класс **обобщённый**: `T` — не конкретный тип, а параметр, который подставит тот, кто классом пользуется. Лотерея из чисел и лотерея из строк — один и тот же класс с разным `T`.

Достаточно читать `T` как «какой-то один и тот же тип»: если в лотерею загрузили числа, то и вынется из неё число.

In [ ]:
import abc
from collections.abc import Iterable


class Tombola[T](abc.ABC):
    @abc.abstractmethod
    def load(self, iterable: Iterable[T]) -> None:
        """Добавить сущности из итерируемого объекта."""

    @abc.abstractmethod
    def pick(self) -> T:
        """Случайно удаляет сущность, возвращая ее.

        Этот метод должен пробрасывать `LookupError` когда инстанс пустой.
        """

    def loaded(self) -> bool:
        """Возвращает `True` если есть хотя бы одна сущность, иначе `False`."""
        return bool(self.inspect())

    def inspect(self) -> tuple[T, ...]:
        """Возвращает кортеж хранящихся сущностей."""
        items: list[T] = []
        while True:
            try:
                items.append(self.pick())
            except LookupError:
                break
        self.load(items)
        return tuple(items)

Мы помечаем метод при помощи `abc.abstractmethod` чтобы показать, что этот метод нужно определить в классе-ребенке. Также мы наследуем ребенку 2 конкретно определенных метода: `loaded` и `inspect`, которые тоже можно переопределять. Код `inspect` не очень умный, но видно, что он полагается на реализованные в детях `pick` и `load`.

<div style="
    background-color: #44944A;
    padding: 15px;
    border-radius: 10px;
    border: 1px solid #fbfbfbff;
    text-align: center;
    display: flex;
    justify-content: center;
    align-items: center;
    min-height: 15px;
">
    <span style="color: white; font-weight: bold;">
        Абстрактный метод всегда должен быть переопределен в классе-ребенке.
    </span>
</div>

In [ ]:
import random


class BingoCage[T](Tombola[T]):
    def __init__(self, items: Iterable[T]) -> None:
        self._randomizer = random.SystemRandom()
        self._items: list[T] = []
        self.load(iterable=items)

    def load(self, iterable: Iterable[T]) -> None:
        self._items.extend(iterable)
        self._randomizer.shuffle(self._items)

    def pick(self) -> T:
        try:
            return self._items.pop()
        except IndexError:
            raise LookupError("pick from empty BingoCage") from None

    def __call__(self) -> T:
        return self.pick()

In [ ]:
bc = BingoCage(items=(i for i in range(5)))

print("поля инстанса:", bc.__dict__)
print("свои методы:  ", [m for m in dir(bc) if not m.startswith("_")])
print("содержимое:   ", bc.inspect())

In [ ]:
bc(), bc.pick()

In [ ]:
bc.load(i for i in range(10, 15))

In [ ]:
while True:
    try:
        print(bc())
    except LookupError as err:
        print(err)
        break

bc.loaded()

In [ ]:
class LottoBlower[T](Tombola[T]):
    def __init__(self, iterable: Iterable[T]) -> None:
        self._balls = list(iterable)

    def load(self, iterable: Iterable[T]) -> None:
        self._balls.extend(iterable)

    def pick(self) -> T:
        try:
            position = random.randrange(len(self._balls))
        except ValueError:
            # randrange на пустом диапазоне бросает ValueError, а не IndexError:
            # без этой ветки наружу ушло бы ValueError и контракт Tombola сломался бы
            raise LookupError("pick from empty LottoBlower") from None
        return self._balls.pop(position)

    def loaded(self) -> bool:
        return bool(self._balls)

    def inspect(self) -> tuple[T, ...]:
        return tuple(self._balls)

    def __call__(self) -> T:
        return self.pick()

In [ ]:
lb = LottoBlower(iterable=(i for i in range(5)))

print("поля инстанса:", lb.__dict__)
print("свои методы:  ", [m for m in dir(lb) if not m.startswith("_")])
print("содержимое:   ", lb.inspect())

In [ ]:
lb(), lb(), lb.inspect()

In [ ]:
class Fake(Tombola[int]): ...


try:
    Fake()
except TypeError:
    print("Класс не реализовывает интерфейс лоттереи")

Данный пример показывает как валидировать пользовательские классы при помощи пользовательского абстрактного класса. Это и есть **интерфейсы**.

## Структурная типизация и статичные протоколы

Типизация при помощи абстрактных классов - номинальная типизация. Наследуя от абстрактного класса мы ставим пометку об этом, после чего `issubclass` смотрит на эту пометку и говорит, является ли класс подклассом некоторой абстракци или нет.

Есть иной подход - **структурная типизация**. В данном случае мы смотрим на публичный интерфейс класса и проверяем есть ли в нем интересующие нас методы.

In [ ]:
class Struggle:
    def __len__(self) -> int: ...

In [ ]:
from collections import abc

assert isinstance(Struggle(), abc.Sized)
assert issubclass(Struggle, abc.Sized)

`issubclass` посчитал класс `Struggle` подклассом абстрактного класса `Sized`, потому что в публичном интерфейсе `Struggle` есть метод `__len__`.

Тут стоит остановиться. Двумя ячейками выше сказано, что `issubclass` смотрит на пометку о наследовании, — а `Struggle` ни от кого не наследуется.

Разгадка в том, что у части абстрактных классов определён специальный метод `__subclasshook__`. Он вызывается при проверке и может ответить «да» просто потому, что у класса есть нужные методы: у `Sized` он ищет `__len__`, у `Iterable` — `__iter__`, у `Hashable` — `__hash__`.

То есть `collections.abc` наполовину структурен: простые протоколы из одного метода распознаются по устройству, а составные вроде `Sequence` — нет.

In [ ]:
print("у Sized хук есть:      ", issubclass(Struggle, abc.Sized))
print("а у Sequence его нет:  ", issubclass(Struggle, abc.Sequence))

Есть и второй способ обойтись без наследования — явно зарегистрировать класс как **виртуального наследника**. Тогда `isinstance` и `issubclass` ответят «да», хотя в родословной класса абстракции нет и ни одного метода он от неё не получит:

In [ ]:
@abc.Sequence.register
class FakeSequence:
    """Ничего не реализует, но зарегистрирован как последовательность."""


print("issubclass говорит да: ", issubclass(FakeSequence, abc.Sequence))
print("а в mro её нет:        ", abc.Sequence in FakeSequence.__mro__)

Отсюда важный вывод: `isinstance` с абстрактным классом отвечает на вопрос «объявлено ли соответствие», а не «работает ли оно на самом деле». Регистрация — обещание, которое никто не проверяет. Ровно поэтому чуть выше и понадобился `fast fail`.

Теперь разберемся с статичными протоколами. Посмотрим на интересный пример:

In [ ]:
from typing import Any


def double(x: Any) -> Any:
    return x * 2

In [ ]:
from fractions import Fraction  # дробь

assert double(1.5) == 3
assert double("a") == "aa"
assert double(Fraction(1, 5)) == Fraction(2, 5)

Понятное дело, что наша функция `double` будет работать с инстансом всякого класса, у которого реализован dunder метод `__mul__`, притом реализация должна работать со скаляром.

До сих пор, чтобы заявить «мой класс поддерживает такой-то интерфейс», нужно было наследоваться от абстрактного класса. Но так можно поступить только со своим кодом: чужой класс не переделать, а сторонняя библиотека о ваших абстракциях не знает.

`typing.Protocol` решает ровно это. Протокол — договор, к которому **не надо присоединяться заранее**: в нём перечислены методы, и подходит ему любой класс, у которого эти методы есть.

In [ ]:
from typing import Protocol


class Printable(Protocol):
    def title(self) -> str: ...

    def lines(self) -> list[str]: ...


class Note:  # обратите внимание: ни от чего не наследуемся
    def __init__(self, title: str, text: str) -> None:
        self._title = title
        self._text = text

    def title(self) -> str:
        return self._title

    def lines(self) -> list[str]:
        return self._text.splitlines()


def render(item: Printable) -> str:
    return "\n".join([item.title(), *(f"  {line}" for line in item.lines())])


print(render(Note("Список дел", "купить хлеб\nпозвонить маме")))

`Note` про `Printable` ничего не знает, и это работает: тайпчекер сам сверил набор методов. Это и есть **структурная типизация** — судят по устройству, а не по родословной.

Но по умолчанию протокол существует только для тайпчекера, в рантайме его проверить нельзя:

In [ ]:
try:
    isinstance(Note("a", "b"), Printable)
except TypeError as err:
    print("Ошибка:", err)

Чтобы `isinstance` заработал, протокол помечают декоратором `@runtime_checkable`. Проверяется при этом только **наличие методов**, но не их сигнатуры — на это полагаться нельзя:

In [ ]:
from typing import runtime_checkable


@runtime_checkable
class RuntimePrintable(Protocol):
    def title(self) -> str: ...

    def lines(self) -> list[str]: ...


print(isinstance(Note("a", "b"), RuntimePrintable))
print(isinstance(42, RuntimePrintable))